In [ ]:
import re

import astropy.units as u
import numpy as np
from astropy.time import Time
from astroquery.jplhorizons import Horizons
from astroquery.mpc import MPC


# Function to get ephemerides from MPC
def mpc_eph(body, epoch):
  
  # Handle minutes
  if re.search(r'm$', epoch['step']):
    epoch['step'] = re.sub(r'm$', 'min', epoch['step'])
  
  # Get ephemerides
  eph = MPC.get_ephemeris(
    body, 
    start=epoch['start'], 
    step=epoch['step'], 
    number=epoch['number']
  )
  eph['Date_jd'] = Time(eph['Date']).jd1 # date must be in JD
  
  # Extract important fields
  data = eph['Date_jd', 'RA', 'Dec', 'Delta']
  error = []
  
  # Check for uncertainty fields
  unc_cols = ['Uncertainty 3sig', 'Unc. P.A.']
  if all(col in eph.colnames for col in unc_cols):
    error = eph['Uncertainty 3sig', 'Unc. P.A.']
  else:
    print('Ephemeris uncertainty is not available.')
  
  # Convert to dataframe
  df = data.to_pandas()
  
  # Return data and error
  return df, error

# Function to get ephemerides from JPL
def jpl_eph(body, epoch):
  
  # Get ephemerides
  body = Horizons(id=body, epochs=epoch)
  eph = body.ephemerides()
  
  # Extract important fields
  data = eph['datetime_jd', 'RA', 'DEC', 'delta']
  error = eph['RA_3sigma', 'DEC_3sigma', 'SMAA_3sigma', 'SMIA_3sigma', 'Theta_3sigma']
  
  # Check if uncertainties exist
  if all(error['RA_3sigma'].mask):
    print('Uncertainty is masked, orbit solution might only be nominal and not have an uncertainty available.')
  
  # Convert to dataframe
  df = data.to_pandas()
  
  # Return data and error
  return df, error

# Parámetros dummies
body_jpl = "2025 OL331"
body_mpc = "2025 NN244"
epoch = {
  "start": str(Time.now()),
  "stop": str(Time.now() + 1 * u.day),
  "step": "1m",
  "number": 720
}

In [ ]:
df, err = jpl_eph(body_jpl, epoch)

In [53]:
err[0]

RA_3sigma,DEC_3sigma,SMAA_3sigma,SMIA_3sigma,Theta_3sigma
arcsec,arcsec,arcsec,arcsec,deg
float64,float64,float64,float64,float64
9.168,2.119,9.407,0.225,12.947


In [61]:
def jpl_unc(err_row, method="sqr"):
  match method:
    case "sqr":
      ra_var = (err_row['RA_3sigma'] / 3) ** 2
      dec_var = (err_row['DEC_3sigma'] / 3) ** 2
      return np.diag([ra_var, dec_var])
    case "cov":
      smaa_var = (err_row['SMAA_3sigma'] / 3) ** 2
      smia_var = (err_row['SMIA_3sigma'] / 3) ** 2
      theta = np.deg2rad(err_row['Theta_3sigma'])
      mat = np.diag([smaa_var, smia_var])
      rot = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
      ])
      return rot @ mat @ rot.T

In [65]:
jpl_unc(err[0], method="sqr")

array([[9.339136  , 0.        ],
       [0.        , 0.49890678]])

In [64]:
jpl_unc(err[0], method="cov")

array([[9.33911834, 2.14571478],
       [2.14571478, 0.4989121 ]])

In [69]:
df, err = mpc_eph(body_mpc, epoch)

In [76]:
err[0]

Uncertainty 3sig,Unc. P.A.
arcsec,deg
int64,float64
42,71.4


In [ ]:
def mpc_unc(err_row, method="sqr"):
  match method:
    case "sqr":
      var = (err_row['Uncertainty 3sig'] / 3) ** 2
      return np.diag([var, var])

In [79]:
mpc_unc(err[0], method="sqr")

array([[196.,   0.],
       [  0., 196.]])